# Batch Product Scraping

This notebook runs the CSV batch scraper. It creates one artifact folder per input row and writes a mapping CSV from input row to artifact paths/status. No search or URL discovery is performed.

In [ ]:
from pathlib import Path
import sys, json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_PATH:', SRC_PATH)

## Optional: clear cached modules

In [ ]:
import sys
for name in list(sys.modules):
    if name.startswith('product_scraping_agent'):
        del sys.modules[name]

from product_scraping_agent import BatchOptions, run_batch
print('Imported batch runner')

## Create or point to an input CSV

Minimum columns: `input_id, product_url`. Recommended columns include product identity and source alignment fields.

In [ ]:
import csv

input_csv = PROJECT_ROOT / 'data' / 'samples' / 'batch_input_sample.csv'
input_csv.parent.mkdir(parents=True, exist_ok=True)

if not input_csv.exists():
    rows = [
        {
            'input_id': 'P001',
            'product_url': 'https://www.amazon.com/Barbie-Collectible-All-Denim-Signature-Underwear/dp/B0BHFD7BPM',
            'main_text': 'THE MOVIE 2023 KEN DOLL WEARING DENIM SET',
            'ean': '194735174539',
            'requested_retailer_name': 'Mercado Libre',
            'requested_country_code': 'CO',
            'source_retailer_name': 'Amazon',
            'source_country_code': 'US',
            'source_url_role': 'global_fallback',
        }
    ]
    with input_csv.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

print(input_csv)
print(input_csv.read_text(encoding='utf-8')[:2000])

## Run batch

In [ ]:
output_csv = PROJECT_ROOT / 'data' / 'batch_scrape_output.csv'
summary_json = PROJECT_ROOT / 'data' / 'batch_scrape_summary.json'

summary = await run_batch(
    input_csv=input_csv,
    output_csv=output_csv,
    summary_json=summary_json,
    options=BatchOptions(
        output_root=PROJECT_ROOT / 'data' / 'scraped',
        max_concurrency=2,
        max_images=30,
        vision_max=12,
        max_agent_iterations=2,
        resume=False,
    ),
)
summary.as_dict()

## Inspect output mapping CSV

In [ ]:
import csv

with output_csv.open('r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))

print('Rows:', len(rows))
for row in rows[:5]:
    print(json.dumps({
        'input_id': row.get('input_id'),
        'success': row.get('success'),
        'artifact_quality': row.get('artifact_quality'),
        'requires_manual_review': row.get('requires_manual_review'),
        'artifact_dir': row.get('artifact_dir'),
        'product_evidence_json_path': row.get('product_evidence_json_path'),
        'claims_md_path': row.get('claims_md_path'),
        'error': row.get('error'),
    }, indent=2, ensure_ascii=False))

## Open first artifact reports

In [ ]:
def read_json(path):
    p = Path(path) if path else None
    return json.loads(p.read_text(encoding='utf-8')) if p and p.exists() else {}

if rows:
    row = rows[0]
    print('QUALITY')
    print(json.dumps(read_json(row.get('quality_report_path')), indent=2, ensure_ascii=False))
    print('\nSOURCE ALIGNMENT')
    print(json.dumps(read_json(row.get('source_alignment_report_path')), indent=2, ensure_ascii=False))
    print('\nCLAIMS MD PREVIEW')
    claims_path = Path(row.get('claims_md_path') or '')
    print(claims_path.read_text(encoding='utf-8')[:4000] if claims_path.exists() else '(missing)')

## Resume example

Use `resume=True` to append and skip input IDs already marked success in the output CSV.

In [ ]:
# summary = await run_batch(
#     input_csv=input_csv,
#     output_csv=output_csv,
#     summary_json=summary_json,
#     options=BatchOptions(
#         output_root=PROJECT_ROOT / 'data' / 'scraped',
#         max_concurrency=2,
#         resume=True,
#     ),
# )
# summary.as_dict()